In [74]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.metrics import classification_report,confusion_matrix
import nltk
from nltk.corpus import stopwords


In [66]:
SEED_STATE = 42

In [67]:
AU_train = pd.read_csv("./BESSTIE/BESSTIE_en_AU_train.csv")
AU_val = pd.read_csv("./BESSTIE/BESSTIE_en_AU_validation.csv")

AU = pd.concat([AU_train, AU_val])
AU.head(5)

,source,variety,text,sentiment,sarcasm
0,google,en-AU,This was one of the best dishes I've EVER had!...,1,0
1,google,en-AU,This Mexican restaurant in Penrith is a great ...,1,0
2,google,en-AU,"This was not to bad, I ordered the big pork ri...",1,0
3,google,en-AU,Clean cool and a nice smaller casino to check ...,1,0
4,google,en-AU,Well set out. Great areas to enjoy. Good food ...,1,0


In [68]:
X,y = AU["text"], AU["sentiment"]

#split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED_STATE,
    stratify=y
)

#divide temporary into 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED_STATE,
    stratify=y_temp
)

In [69]:
model = Pipeline([
    ("vectorizer", CountVectorizer(
        lowercase=True,max_features=20000
    )),
    ("classifier", MultinomialNB(alpha=1.0))
])

model.fit(X_train, y_train)
val_pred = model.predict(X_val)

In [70]:
# param_grid = {
#     "vectorizer": [
#         CountVectorizer(),
#         TfidfVectorizer()
#     ],
#     "vectorizer__ngram_range": [(1, 1), (1, 2)],
#     "vectorizer__min_df": [1, 2, 5],
#     "vectorizer__max_features": [10000, 20000],
#     "classifier__alpha": [0.1, 0.5, 1.0, 2.0]
# }

# grid = GridSearchCV(
#     model,
#     param_grid,
#     cv=5,
#     scoring="f1_macro",
#     n_jobs=-1
# )

# grid.fit(X_train, y_train)

# print(grid.best_params_)
# print(grid.best_score_)

In [87]:
model2 = Pipeline([
    ("vectorizer", TfidfVectorizer(
        analyzer="char_wb",
        min_df=2,
        max_features=50000
    )),
    ("classifier", MultinomialNB(alpha=0.1))
])

model2.fit(X_train, y_train)
val2_pred = model2.predict(X_val)

In [88]:
report = classification_report(y_val, val2_pred)
print(report)

              precision    recall  f1-score   support

           0       0.56      1.00      0.72       247
           1       0.95      0.10      0.18       215

    accuracy                           0.58       462
   macro avg       0.76      0.55      0.45       462
weighted avg       0.74      0.58      0.47       462



In [73]:
report = classification_report(y_val, val_pred)
print(report)

              precision    recall  f1-score   support

           0       0.80      0.83      0.82       247
           1       0.80      0.77      0.78       215

    accuracy                           0.80       462
   macro avg       0.80      0.80      0.80       462
weighted avg       0.80      0.80      0.80       462

